# Dados a serem coletados

- Nome -> Steam - Ok
- AppID -> http://api.steampowered.com/ISteamApps/GetAppList/v2/
- Preço -> Steam - Ok

- Quantidade de avaliação -> Steam - OK
- Lançamento -> Steam - OK
- Idade recomendada -> Steam - 1/2 Ok (parece que alguns jogos não tem essa info)

- Duração do jogo -> How Long To beat

- Marcadores/Tags -> SteamSpy
- Avaliação dos usuário -> SteamSpy

- Free - Steam - Ok
---
- Acesso antecipado - Steam -> É um gênero, podendo ser filtrado por isso
- DLC - Steam -Ok
- Lançamento futuro - Steam -> 'release_date': {'coming_soon': True, 'date': 'Q1 2024'}
- Línguas -> Steam - Ok
- Características do jogo -> Steam - Ok
- Publicadora -> Steam - Ok
- Desenvolvedora -> Steam - OK
- Gênero -> Steam - Ok
- Conquistas (quantidade) -> Steam - OK

- Item na loja (quantidade) -> Steam ?

- Avaliação Crítica -> Metacritic, OpenCritic, Steam - 1/2 OK (Só tem metacritic pelo steam) 
- Requisitos Mínimos (Windows) -> Steam - Ok
- Requisitos Máximos (Windows) -> Steam - OK

# HTMLs

- Características: class="game_area_features_list_ctn">


### Coleta de dados por jogo

In [31]:
import requests as rs
import json

url = "http://store.steampowered.com/api/appdetails/"

appIds = []

appIdTest = 435150
dlcAppId = 715950 #-> Testar se conseguimos diferenciar DLC de jogo

currency='us'
language='en'

params = {"appids": appIdTest, "cc": currency, "l": language}

response = rs.get(url=url, params=params, timeout=10)
data = response.json()['435150']['data']
for a in data:
    print(a)
#print(response.json()['435150']['data']['price_overview'])



type
name
steam_appid
required_age
is_free
controller_support
dlc
detailed_description
about_the_game
short_description
supported_languages
reviews
header_image
capsule_image
capsule_imagev5
website
pc_requirements
mac_requirements
linux_requirements
legal_notice
developers
publishers
price_overview
packages
package_groups
platforms
metacritic
categories
genres
screenshots
movies
recommendations
achievements
release_date
support_info
background
background_raw
content_descriptors


### Coleta de AppIds

In [29]:
import requests as rs
import json

url_AppIds = "http://api.steampowered.com/ISteamApps/GetAppList/v2/"

response = rs.get(url=url_AppIds,timeout = 10)
if response:
    data = response.json()
    appIds = data['applist']['apps'] #Apps é uma lista de dicionários, com chaves "appid" e "name"
    with open('Steam_AppIds.json','w') as file:
        json.dump(appIds,file,indent=4)
    print(f'Arquivo de appIDs foi criado com {len(appIds)} IDs')
else:
    print('A API não retornou resposta')

Arquivo de appIDs foi criado com 179186 IDs


In [38]:
import json

appIds = []

with open('Steam_AppIds.json','r') as file:
    appIds = json.load(file)

appIdsRange = int(len(appIds)/100)

print(appIdsRange)

currency='us'
language='en'

count = 0
for s in range(0,100): # Testar com apenas 1 segmento primeiro
    if s == 99:
        print(len(appIds[appIdsRange*s:len(appIds)]))
        count += len(appIds[appIdsRange*s:len(appIds)])
    else:
        count += len(appIds[appIdsRange*s:appIdsRange*(s+1)])
print (count == len(appIds))

1791
1877
True


### Funções úteis

In [13]:

import time

columnsToIgnore =['dlc', 'detailed_description', 'about_the_game', 'reviews', 'header_image', 'capsule_image', 'capsule_imagev5', 'website' , 'mac_requirements', 
                  'linux_requirements', 'legal_notice', 'packages', 'package_groups', 'screenshots', 'movies', 'support_info', 'background', 'background_raw', 'content_descriptors']

def RemoveUselessData(data):
    for k in columnsToIgnore: 
        try:
            data.pop(k)
        except:
            continue

currency='us'
language='en'

def GetData(appid,name):
    try:
        response = rs.get(url=url, params=params, timeout=10)
        if response:
            try:
                data = response.json()[str(app['appid'])]['data'] # => É um dicionário
                for k in columnsToIgnore: 
                    try:
                        data.pop(k)
                    except:
                        continue
                data['scrap_status'] = 'Scrap_Sucess'
            except KeyError:
                data = {'appid':appid,'name':name,'scrap_status':'Scrap_NoData'}
        elif response.status_code == 429:
            print('Esperando api')
            time.sleep(5)
            data = GetData(appid,name)
        else:
            data = {'appid':appid,'name':name,'scrap_status': f'Scrap_Fail {response}'}
    except TimeoutError as te:
        data = {'appid':appid,'name':name,'scrap_status':'Scrap_TimeoutError'}
    except Exception as e:
        data = {'appid':appid,'name':name,'scrap_status':f'Scrap_Error {type(e)}'}
    return data

### Scrap em segmentos/partes de toda a Steam

In [12]:
import requests as rs
import json
from tqdm.notebook import trange, tqdm

url = "http://store.steampowered.com/api/appdetails/"

appIds = []

with open('Steam_AppIds.json','r') as file:
    appIds = json.load(file)

appIdsRange = int(len(appIds)/100)

currency='us'
language='en'

columnsToIgnore =['dlc', 'detailed_description', 'about_the_game', 'reviews', 'header_image', 'capsule_image', 'capsule_imagev5', 'website' , 'mac_requirements', 
                  'linux_requirements', 'legal_notice', 'packages', 'package_groups', 'screenshots', 'movies', 'support_info', 'background', 'background_raw', 'content_descriptors']

for s in range(0,1): # Testar com apenas 1 segmento primeiro
#for s in range(0,100):
    if s == 99:
        segment = appIds[appIdsRange*s:len(appIds)]
    else:
        segment = appIds[appIdsRange*s:appIdsRange*(s+1)]
    dataList = []
    progresso = tqdm(total=1, desc=f"Progresso do {s} segmento", position=0, leave=True)
    for app in segment:
        params = {"appids": app['appid'], "cc": currency, "l": language}
        try:
            response = rs.get(url=url, params=params, timeout=10)
            if response:
                try:
                    data = response.json()[str(app['appid'])]['data'] # => É um dicionário
                    for k in columnsToIgnore: 
                        try:
                            data.pop(k)
                        except:
                            continue
                    data['scrap_status'] = 'Scrap_Sucess'
                except KeyError:
                    data = {'appid':app['appid'],'name':app['name'],'scrap_status':'Scrap_NoData'}
            elif response.status_code == 429:
                print 
            else:
                data = {'appid':app['appid'],'name':app['name'],'scrap_status': f'Scrap_Fail {response}'}
        except TimeoutError as te:
            data = {'appid':app['appid'],'name':app['name'],'scrap_status':'Scrap_TimeoutError'}
        except Exception as e:
            data = {'appid':app['appid'],'name':app['name'],'scrap_status':f'Scrap_Error {type(e)}'}
        dataList.append(data)
        progresso.update(1/len(segment))
    with open(f'RawData/SteamAppData_{s}.json','w') as file:
        json.dump(dataList,file,indent=4)
    print(f'Pacotes de dados coletados: {s+1}/100')

Progresso do 0 segmento:   0%|          | 0/1 [00:00<?, ?it/s]

Pacotes de dados coletados: 1/100


### Tentando pegar os dados denovo das coletas que falharam

In [14]:
import json
from tqdm.notebook import trange, tqdm

with open(f'RawData/SteamAppData_{0}.json','r') as file:
    data = json.load(file)

noDataCount = 0
progresso = tqdm(total=1, desc=f"Progresso do {s} segmento", position=0, leave=True)

for i,app in enumerate(data):##list(filter(lambda a: 'Scrap_Sucess' not in a.get('scrap_status'), data)):
    status = app['scrap_status']
    if "Scrap_Fail" in status or "Scrap_TimeoutError" in status or 'Scrap_Error' in status:
        data[i] = GetData(app['appid'],app['name'])
    else:
        noDataCount+=1
print(noDataCount)
with open(f'RawData/SteamAppDataComplete_{0}.json','w') as file:
    json.dump(data,file,indent=4)
    

UnsupportedOperation: not readable

In [48]:
import requests as rs
import json

url = "http://store.steampowered.com/api/appdetails/"

appIds = []

appIdTest = 606150
dlcAppId = 715950 #-> Testar se conseguimos diferenciar DLC de jogo

currency='us'
language='en'

params = {"appids": appIdTest, "cc": currency, "l": language}

response = rs.get(url=url, params=params, timeout=10)
data = response.json()[str(appIdTest)]['data']
#print(data.keys())

with open(f'RawData/SteamAppData_{0}.json','w') as file:
    json.dump(data,file,indent=4)

In [44]:
import requests as rs
import json

url = "http://store.steampowered.com/api/appdetails/"

appIds = []

appIdTest = 1941401
dlcAppId = 715950 #-> Testar se conseguimos diferenciar DLC de jogo

currency='us'
language='en'

params = {"appids": appIdTest, "cc": currency, "l": language}

response = rs.get(url=url, params=params, timeout=10)
data = response.json()[str(appIdTest)]
print(data.keys())

dict_keys(['success'])


### Amostras para testar todo o processo
- Moonlighter = 606150
- Divinity Original Sin 2 = 435150
- Zomboid = 108600
- S.T.A.L.K.E.R. 2: Heart of Chornobyl = 1643320

### Adicionar novos apps

Motivo de existência: Para atualizar o dataset, provavelmente não vai fazer sentido coletar dados de DLCs e etc. dessa forma, primeiro é necessário adicionar os novos APPs para poder filtrar isso e só pegar os dados que realmente interessam, a fim de reduzir substancialmente o tempo para cada coleta de dados

1. Atualizar a lista de APPIDs
2. Ler o arquivo gerado e subtrair os APPs já conhecidos pelo dataset
3. Coletar os dados dos novos APPs e anexar ao dataset